In [2]:
from pathlib import Path

dir_evals = Path('docs/temp/evals')
if not dir_evals.exists(): print(f"Directory {dir_evals} does not exist.")

tools = ['chatgpt_deepresearch', 'manus_ai', 'discourse2draft']

In [3]:
import re
from src.backend.ai.llms import getAIModel
from pydantic import BaseModel, Field
from langchain.output_parsers.fix import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from src.backend.ai.prompts import setPrompt
from src.backend.utils import Config
import pandas as pd

class StandardizeSectionHeadersSchema(BaseModel):
    '''
    Returns the mapping from the section header to standard section header
    '''
    mapping: dict[str, str] = Field(description='A mapping from the section header to standard section header')

def standardizeHeader(section_headers, section_headers_standard):

    llm = getAIModel(model_name='azure-gpt-4o')
    system_prompt = 'You are an AI assist designed to standardize section headers in a document'
    human_prompt = '''
                You will be provided with a document which is a list of section headers.
                You will also be provided with a list of standard section headers.
                Your task is to map each provided section header to a standard section header.
                If there is no mapping possible, map the section header to itself.

                <Section headers>
                {section_headers}
                </Section headers>

                <Standard section headers>
                {section_headers_standard}
                </Standard section headers>
                '''
    
    parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=StandardizeSectionHeadersSchema), 
                                llm=llm,
                                max_retries=Config.RETRY_COUNTER)
    
    prompt = setPrompt(system_prompt, human_prompt, parser)
        
    response = (prompt | llm | parser).invoke(input={'section_headers': section_headers, 'section_headers_standard': section_headers_standard})
    return dict(response)['mapping']

def extractSectionContent(file_path):
    
    with open(file_path) as fp:
        data = '\n'.join([line.strip() for line in fp.readlines()])
    
    output = re.findall(r'(#+ .+\n?)([^#]+)', data)
    return [(section.strip(), content.strip()) for section, content in output]

def getSectionAndContent(file_path, section_headers_standard):
    doc = extractSectionContent(file_path)
    doc_san = []
    for section, content in doc:
        if section.startswith('# '):
            doc_san.append(['# Title', section[2:]])
        elif section.startswith('## '):
            doc_san.append([section, content])
        else:
            doc_san[-1][1] += section + '\n' + content
    section_headers = [section for section, _ in doc_san]
    mapping = standardizeHeader(section_headers, section_headers_standard)
    return [(mapping.get(section, section), content) for section, content in doc_san]
        
def getSectionsForComparison(file_name):

    section_headers_standard = [item for item in extractSectionContent(dir_evals / 'prompts' / file_name) if not item[0].startswith('# ')]
    responses = {tool: dict(getSectionAndContent(dir_evals / tool / file_name, section_headers_standard)) for tool in tools}
    
    return pd.DataFrame(responses)


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers.fix import OutputFixingParser
from pydantic import BaseModel, Field
from src.backend.ai.prompts import setPrompt
from src.backend.ai.common import StateContentManager
from src.backend.utils import Config

class ScoreWithReasonSchema(BaseModel):
    '''
    Returns a score within 0 to 100 for a particular criterion and a statement supporting the score
    '''
    score: int = Field(description='Score within 0 to 100')
    reason: str = Field(description='A short statement supporting the score')

class RateContentSchema(BaseModel):
    '''
    Returns scores based on different criteria to rate the content of a structured document
    '''

    relevance: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Relevance of the content')
    continuity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Continuity of the content')
    non_repetitiveness: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Uniqueness (or non-repetitiveness) of the content')
    specificity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Specificity of the content')
    #relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

# ---------------------------------------------------------------------------
class RateContent:

    rate_content_system_prompt = '''\
    You will be provided a manuscript section header with or without previously written content summary on a specific topic. You are a scholarly reviewer with expertise in the corresponding topic area. Your task is review and rate the section content.
    
    <Instructions>
    - Rating must be an integer number within 0 (lowest) and 100 (highest).
    - Rating will base on the following criteria.
    **Relevance:** Is the content relevant to the provided section header and the summary (if provided).
    **Continuity:** Is the content writing style is continuous with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Non-Repetitiveness:** Is the content devoid of repetitiveness compared with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Specificity:** Is the content devoid of halucination.
    </Instructions>
    '''

    rate_content_human_prompt = '''
    <Previous Content Summary>
    {content_pre}
    </Previous Content Summary>

    <Current Section Header>
    {current_section}
    </Current Section Header>

    <Content>
    {content}
    </Content>
    
    <Instructions>
    - Read the Previous Content Summary. 
    - Review the provided content based on the Current Section Header
    - Provide the output in the following format.
    {format_instructions}
    
    - Output must be in JSON format with `json` tags.
    </Instructions>
    '''

    def __init__(self, llm):

        parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=RateContentSchema), 
                                             llm=llm,
                                             max_retries=Config.RETRY_COUNTER)
        
        self.rate_content_prompt = setPrompt(self.rate_content_system_prompt, self.rate_content_human_prompt, parser)
        
        self.rate_content_chain = self.rate_content_prompt | llm | parser


    def __call__(self, state: StateContentManager):
        '''LLM generates reports from a given outline'''
        
        response = self.rate_content_chain.invoke(input={'content_pre': state['content_pre'],
                                                             'current_section': state['current_section'],
                                                             'content': state['content']})
        try:
            content = dict(response)
        except:
            raise Exception(f'GenerateContent response does not have content, response: {response}')

        return {'content': content, 'steps': ['Rate Content']}
    
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph
from src.backend.ai.llms import getAIModel
from src.backend.ai.summarize import Summarize
from typing import Literal

# -----------------------------------------------------------------------
def check_if_summary_needed(
        state: StateContentManager,
    ) -> Literal['Summarize', 'Rate Content']:
        if len(state.get('content_pre').split()) > 500:
            return 'Summarize'
        return 'Rate Content'

# -----------------------------------------------------------------------
class ContentWriterArchitecture:
     
     def __init__(self, model_name, temperature):
        llm = getAIModel(model_name=model_name, temperature=temperature)

        # Define a new graph
        workflow = StateGraph(state_schema=StateContentManager)

        # Define the (single) node in the graph
        workflow.add_node("Summarize", Summarize(llm=llm, input_field='content_pre'))
        workflow.add_node("Rate Content", RateContent(llm=llm))
        #workflow.add_node("Add Citations", AddCitations(llm))

        workflow.add_conditional_edges(START, check_if_summary_needed)
        workflow.add_edge("Summarize", "Rate Content")

        self.agent = workflow.compile()


In [6]:
file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    display(getSectionsForComparison(file_name))

Processing "CRISPR-based editing for inherited blood disorders.md"...


Calling src.backend.ai.llms.getAIModel

2026-01-22 09:18:50 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:18:50 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-22 09:18:52 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-22 09:18:54 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


...

Calling src.backend.ai.llms.getAIModel

2026-01-22 09:24:37 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:24:37 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-22 09:24:39 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-22 09:24:41 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


,chatgpt_deepresearch,manus_ai,discourse2draft
# Title,Health Impacts of Per- and Polyfluoroalkyl Sub...,Health Impacts of Per- and Polyfluoroalkyl Sub...,PFAS
## Abstract:,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFAS) are...
## Introduction & Chemical Background:,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFAS) are...
## Exposure Pathways & Toxicokinetics:,**Exposure Pathways:** Human exposure to PFASs...,### 2.1. Exposure Pathways\nHuman exposure to ...,Drinking water contamination has emerged as a ...
## System-Specific Health Impacts (The Core Analysis):,### Immunotoxicity\nOne of the most sensitive ...,The evaluation of health impacts is based on t...,Immunotoxicity. There is sufficient evidence t...
## Mechanisms of Action:,Elucidating how PFASs cause the array of obser...,The biological plausibility of the observed he...,PFAS interactions with peroxisome proliferator...
"## The ""GenX"" Problem:","By the early 2000s, scientific and regulatory ...",The phase-out of PFOA and PFOS led to the intr...,In response to regulatory pressure and steward...
## Regulatory Context & Knowledge Gaps:,**Regulatory Context:** Confronted with mounti...,### 6.1. Regulatory Context\nRegulatory bodies...,"In March 2023, the U.S. Environmental Protecti..."
## Conclusion:,PFASs have transitioned from scientific obscur...,The weight of evidence from both epidemiologic...,"Despite phasedown of legacy long-chain PFAS, t..."
## References,1. **American Bar Association (ABA)**. (2022)....,[1]: https://pmc.ncbi.nlm.nih.gov/articles/PMC...,"[1]: Mazumder, N., Hossain, M. T., Jahura, F. ..."


In [8]:
agent = ContentWriterArchitecture(model_name='azure-o3', temperature=0).agent

Calling src.backend.ai.llms.getAIModel

2026-01-22 09:26:38 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:26:38 [INFO] Calling src.backend.ai.prompts.setPrompt


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:26:38 [INFO] Calling src.backend.ai.prompts.setPrompt


In [ ]:
import tqdm
import pandas as pd

# relevance, continuity, uniqueness, relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

from pathlib import Path

def formatRating(rating, tool):
    rating_dict = {}
    for criterion in rating:
        rating_dict[f'{tool} {criterion} (score)'] = rating[criterion].score
        rating_dict[f'{tool} {criterion} (reason)'] = rating[criterion].reason

    return rating_dict

file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    section_sets = getSectionsForComparison(file_name)
    content_pre_sets = ['' for _ in section_sets]
    rating_responses = {}
    for index, row in tqdm.tqdm(section_sets.iterrows()):
        
        section_header = row.name
        print(section_header)
        
        rating_respons_section_wise = {}
        for i, tool in enumerate(tools):
            
            print(tool)

            content = row[tool] if not pd.isna(row[tool]) else ''

            if content.strip() == '': continue
            
            rating_response = agent.invoke({'current_section': section_header, 'content': content, 'content_pre': content_pre_sets[i]})['content']
            rating_respons_section_wise |= formatRating(rating_response, tool)

            content_pre_sets[i] += f'{section_header}\n\n{content}'

        rating_responses[section_header] = rating_respons_section_wise

    rating_responses = pd.DataFrame(rating_responses)

    rating_score = rating_responses.loc[rating_responses.index.str.endswith('(score)')]
    rating_reason = rating_responses.loc[rating_responses.index.str.endswith('(reason)')]

    rating_score = rating_score.rename(index=lambda x:x.replace(' (score)', ''))
    rating_reason = rating_reason.rename(index=lambda x:x.replace(' (reason)', ''))

    rating_score.to_csv(dir_evals / 'scores' / f'{Path(file_name).stem}.csv', index=True)
    rating_reason.to_csv(dir_evals / 'reasons' / f'{Path(file_name).stem}.csv', index=True)

Processing "CRISPR-based editing for inherited blood disorders.md"...


Calling src.backend.ai.llms.getAIModel

2026-01-22 09:58:20 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:58:20 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-22 09:58:22 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-22 09:58:23 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


...

2026-01-22 09:58:23 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-22 09:58:27 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-22 09:58:28 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


Calling src.backend.ai.llms.getAIModel

2026-01-22 09:58:28 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-22 09:58:28 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-22 09:58:30 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


In [17]:
rating_score.rename(index=lambda x:x.replace(' (score)', ''))

,# Title,## **1. Executive Summary**,## **2. Classification and Exposure Assessment**,## **3. Toxicokinetics (ADME)**,## **4. Molecular Mechanisms of Action**,## **5. Organ-Specific Toxicity (Evidence Review)**,## **6. Vulnerable Populations**,## **7. Regulatory Landscape & Tolerable Limits**,## **8. Emerging Science & Alternatives**,## **9. Conclusion & Future Directions**,## References
chatgpt_deepresearch relevance,90,95,90,90,90,90,85,90,80,82,NaN
chatgpt_deepresearch continuity,100,98,85,88,85,95,80,85,85,75,NaN
chatgpt_deepresearch non_repetitiveness,100,98,95,95,80,92,85,90,90,92,NaN
chatgpt_deepresearch specificity,85,90,88,85,88,88,90,75,75,80,NaN
manus_ai relevance,20,60,95,90,90,90,90,93,90,92,90
manus_ai continuity,100,80,90,85,88,85,85,90,85,88,85
manus_ai non_repetitiveness,100,90,85,80,92,80,80,96,95,90,95
manus_ai specificity,70,78,90,88,85,85,85,88,82,85,80
discourse2draft relevance,40,95,92,92,92,80,90,90,90,92,85
discourse2draft continuity,90,95,85,88,88,85,85,85,85,87,90


In [43]:
rating_responses = pd.DataFrame(rating_responses)

rating_responses.loc[rating_responses.index.str.endswith('(reason)')]

,# Title,## Executive Summary
chatgpt_deepresearch relevance (reason),"The section header is '# Title', and the provi...","Concise overview touches on clinical context, ..."
chatgpt_deepresearch continuity (reason),"No previous content summary exists, so continu...","Flows naturally from the report title, providi..."
chatgpt_deepresearch non_repetitiveness (reason),"The content consists of a single, non-repetiti...",No redundancy detected within this section and...
chatgpt_deepresearch specificity (reason),The title precisely identifies CRISPR/Cas9 and...,"Includes concrete details (exa-cel mechanism, ..."
manus_ai relevance (reason),The section header indicates this should be th...,"Discusses HSC gene-editing therapy, which is t..."
manus_ai continuity (reason),"No previous content summary exists, so continu...","With only the manuscript title available, styl..."
manus_ai non_repetitiveness (reason),"With no earlier content to repeat, the provide...",No internal repetition and no earlier content ...
manus_ai specificity (reason),"Content is concise and factual; however, inclu...","Provides accurate, concrete details (e.g., Exa..."
discourse2draft relevance (reason),The content is clearly a title and matches the...,Concise overview directly addresses CRISPR-bas...
discourse2draft continuity (reason),"No previous content was provided, so continuit...",No prior executive-summary text to match; tone...


[('# CRISPR-based therapeutic genome editing for inherited blood disorders\n## Therapeutic challenges\n### Chemotherapy-free cell therapies',
  ['\nAutologous gene therapies, including exa-cel, typically rely on myeloablative conditioning to facilitate the engraftment of engineered HSPCs12,168,173. Although non-myeloablative chemotherapy can be sufficient for gene-corrected cells with a selective advantage to reach therapeutic levels, myeloablation is necessary for therapeutic indications dependent on high levels of engraftment, such as β-haemoglobinopathies and lysosomal storage disorders217. HSC gene therapies are therefore limited to the most severe forms of blood disorders, because genotoxic conditioning prior to transplantation is associated with acute toxicities such as cytopenias, mucositis, infection and organ injury (requiring inpatient support for 1–2\u2009months), as well as long-term toxicities such as infertility, organ toxicity and risk of secondary malignancy134,166. In 

In [46]:
row

chatgpt_deepresearch    **Adenine Base Editor (ABE):** An engineered e...
manus_ai                | Term            | Definition                ...
discourse2draft                                                       NaN
Name: ## Key Terminology, dtype: object

100%|██████████| 30/30 [37:00<00:00, 74.02s/it] 
